In [68]:
import numpy as np
import scipy.sparse as sp

from math import sqrt

import igl
import pyvista as pv
pv.set_jupyter_backend('trame')
#pv.set_jupyter_backend('static')

In [69]:
def to_pyvista_mesh(V, F):
    return pv.UnstructuredGrid({pv.CellType.TRIANGLE: F}, V)

In [70]:
# each idx corresponds to a face, each element is a component, highest component count = biggest component\
# Filter out noise components
def filter_components(v, f):
    (num_components, component_ids) = igl.facet_components(f)
    unique, counts = np.unique(component_ids, return_counts=True)
    count_per_component = dict(zip(unique, counts))
    
    comp_vals = list(count_per_component.values())
    largest_comp_idx = comp_vals.index(max(comp_vals))
    filtered_f_idx = [i for i, comp in enumerate(component_ids) if comp == largest_comp_idx]
    
    filtered_f = f[filtered_f_idx, :]
    
    to_pyvista_mesh(v, filtered_f).plot(show_edges=True)
    v, f, _, _ = igl.remove_unreferenced(v, f)
    return v, f

In [71]:
def non_zero_cot(angle):
    tan = np.tan(angle)
    if abs(tan) > 0:
        return 1 / tan
    return 0.0

In [72]:
def integrate_heat_flow(v, f, starting_point):
    face_areas = 1/2 * igl.doublearea(v, f)
    angles = igl.internal_angles(v, f)
    
    t = igl.avg_edge_length(v, f) ** 2 # Paper gives t = h^2 as a good 
    delta = np.zeros(len(v))
    delta[starting_point] = 1 # Initialize starting point to vertex zero
    
    vf, ni = igl.vertex_triangle_adjacency(f, len(v))
    A_flat = []
    data = []
    ii = []
    jj = []
    
    L_c = np.zeros((len(v), len(v)))
    for i in range(len(v)):
        adj_faces = []
        diag_val = 0
        # Find all incident faces for current vertex
        for j in range(0, ni[i+1] - ni[i]):
            cur_face = vf[ni[i] + j]
            adj_faces.append(cur_face)
            
            # Find a_ij and b_ij for i
            assoc_tri_verts = f[cur_face]
            angle_idx = np.where(assoc_tri_verts != i)[0]
    
            # L_c at [i, adj_vert a or b] = angle a_ij, b_ij
            a_ij = 0.5 * non_zero_cot(angles[cur_face, angle_idx[1]])
            b_ij = 0.5 * non_zero_cot(angles[cur_face, angle_idx[0]])
    
            diag_val += a_ij + b_ij
        
            ii.append(i); jj.append(assoc_tri_verts[angle_idx[0]]); data.append(a_ij)
            ii.append(i); jj.append(assoc_tri_verts[angle_idx[1]]); data.append(b_ij)
            
        ii.append(i); jj.append(i); data.append(-diag_val)
        A_i = 1/3 * sum(face_areas[adj_faces])
        
        A_flat.append(A_i)
        
    assert(len(A_flat) == len(v))
    
    L_c = sp.coo_matrix((data, (ii, jj)), shape=(len(v), len(v))).asformat("csr")
    A = sp.diags(A_flat, format='csr')
    
    u = sp.linalg.spsolve(A - t * L_c, delta)
    
    assert(len(u) == len(v))
    
    return L_c, u

In [73]:
def eval_vec_field(v, f, u):
    eps=1e-12
    face_areas = 1/2 * igl.doublearea(v, f)
    angles = igl.internal_angles(v, f)
    
    vf, ni = igl.vertex_triangle_adjacency(f, len(v))
    X = np.zeros((len(f), 3))
    face_normals = igl.per_face_normals(v, f)
    
    # Compute gradient of u
    for face_idx, face_verts in enumerate(f):
        cur_face_normal = face_normals[face_idx]
        a, b, c = face_verts
    
        # Compute u_i * (N x e_i) explicitely
        sigma = (u[a] * np.cross(cur_face_normal, v[c] - v[b]) +
                 u[b] * np.cross(cur_face_normal, v[a] - v[c]) +
                 u[c] * np.cross(cur_face_normal, v[b] - v[a]))
    
        grad_u = 1 / (2 * face_areas[face_idx] + eps) * sigma
        norm = np.linalg.norm(grad_u)
        
        # ensure no division by 0
        if norm > 0:
            X[face_idx] = -grad_u / norm
            
    # Compute divergence of X 
    div_X = np.zeros(len(v))
    for i in range(len(v)):
        adj_faces = []
    
        cur_sum = 0
        # Find all incident faces for current vertex
        for j in range(0, ni[i+1] - ni[i]):
            cur_face = vf[ni[i] + j]
            X_j = X[cur_face]
    
            # Find a_ij and b_ij for i
            assoc_tri_verts = f[cur_face]
            angle_idx = np.where(assoc_tri_verts != i)[0]
            
            # theta_1 and theta_2 are opposing angles of i 
            theta_1 = angles[cur_face, angle_idx[1]]
            theta_2 = angles[cur_face, angle_idx[0]]
    
            # e_1 and e_2 are edge vectors containing i
            e_1 = v[assoc_tri_verts[angle_idx[0]]] - v[i]
            e_2 = v[assoc_tri_verts[angle_idx[1]]] - v[i]
                
            cur_sum += non_zero_cot(theta_1) * (np.dot(e_1, X_j)) + non_zero_cot(theta_2) * (np.dot(e_2, X_j))
                
        div_X[i] = 1/2 * cur_sum
        
    return div_X

In [74]:
def solve_poisson(L_c, div_X):
    # Pin as solution is only defined up to additive constant
    b = div_X
    L_c_pinned = L_c.tolil()
    L_c_pinned[0, :] = 0
    L_c_pinned[0, 0] = 1
    L_c_pinned = L_c_pinned.tocsr()
    
    b_pinned = b.copy()
    b_pinned[0] = 0
    
    phi = sp.linalg.spsolve(L_c_pinned, b_pinned)
    
    # Normalize values to 0
    phi -= phi.min()
    
    return phi

In [75]:
def plot_heat_flow(v, f, phi, starting_point):
    pl = pv.Plotter()
    mesh = to_pyvista_mesh(v, f)
    
    # Visualize source point
    pl.add_points(mesh.points[starting_point], color='red', point_size=20)
    
    pl.add_mesh(mesh, scalars=phi, cmap='inferno_r', show_edges=False)
    
    # Plot isolines 
    n_contours = 20
    contours = mesh.contour(
        isosurfaces=np.linspace(phi.min(), phi.max(), n_contours),
        scalars=phi
    )
    
    pl.add_mesh(contours, color='white', line_width=3)
    pl.show()

In [76]:
def compute_heat_flow(mesh_dir, starting_point):
    v, f = igl.read_triangle_mesh(mesh_dir)
    
    # Mesh clean up
    v, _, _, f = igl.remove_duplicate_vertices(v, f, 1e-10)
    areas = igl.doublearea(v, f)
    proper_faces = areas > 1e-15
    f = f[proper_faces]

    used_faces = np.unique(f)
    new_faces = np.full(len(v), -1, dtype=int)
    new_faces[used_faces] = np.arange(len(used_faces))

    v = v[used_faces]
    f = new_faces[f]
    
    v, f = filter_components(v, f)
    
    # Step 1:
    L_c, u = integrate_heat_flow(v, f, starting_point)

    # Step 2: 
    div_X = eval_vec_field(v, f, u)

    # Step 3:
    phi = solve_poisson(L_c, div_X)
    
    plot_heat_flow(v, f, phi, starting_point)

In [77]:
compute_heat_flow("models/cat.off", 100)

Widget(value='<iframe src="http://localhost:57014/index.html?ui=P_0x336ac5f10_16&reconnect=auto" class="pyvist…

Widget(value='<iframe src="http://localhost:57014/index.html?ui=P_0x3234ea5a0_17&reconnect=auto" class="pyvist…